Imports

In [22]:
import sys
import os
package_path = os.path.abspath("../..")  
sys.path.insert(0, package_path)
#The path will be managed by conda or whatever on release, but this is fine for now.
import scMPRAforge as scm
import pandas as pd
import numpy as np

In [1]:
#load the autoreload extension
%load_ext autoreload
#reload code on every execution
#(this may break objects)
#you can remove this & do dev in a notebook, then paste into the module when you are done.
%autoreload 2

In [2]:
from dask_jobqueue import SLURMCluster
from dask.distributed import Client
import socket

In [3]:
scm.helloworld()

hello world!


Make the dask cluster & client in accordance with resource avail and model size

In [58]:
cluster=SLURMCluster(
    cores=2,#cores per slurm job
    memory="16G",#memory per slurm job
    processes=1,#dask workers per slurm job
    job_extra_directives=["-p ycga", 
        f"--job-name=simclust_worker",
        f"--time=2:00:00",
        f"--output=slave_%j.out"]
)
client = Client(cluster,
    timeout=f"{5*60}s",   # Client <-> scheduler timeout 
    heartbeat_interval="20s"  # Worker heartbeat interval
)

In [59]:
cluster.scale(jobs=1)

In [60]:
dat=scm.load_scMPRA_data("/gpfs/gibbs/pi/reilly/tabula_data/simulated/fake_cres.tsv")

In [61]:
#let's make an ortho object
test=scm.ortho()
test.criss_cross(client=client,dat=dat,retain_design_matricies=True)
test.extract_params(client)

In [82]:
cell_counts=scm.get_cell_counts(client,dat,split="cre_id")

flattened_param=scm.flatten_param_representation(client,test.by_cre_parameters.result(),split="cre_id")

In [83]:
flattened_param

nb  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id                  
1                   0                   1            0            0            nobody       1.775819   
                                        0            1            0            nobody       1.775819   
                                                     0            1            nobody       1.775819   
0                   1                   1            0            0            nobody       0.930733   
                                        0            1            0            nobody       0.930733   
                                                     0            1            nobody       0.930733   
1                   0                   1            0            0            somebody    12.475355   
                                        0            1            0            somebody    12.475355   
                                                     0            1            somebody    12.475355   
0                   1                   1            0            0            somebody     9.958996   
                                        0            1            0            somebody     9.958996   
                                                     0            1            somebody     9.958996   
1                   0                   1            0            0            everybody  103.580827   
                                        0            1            0            everybody  103.580827   
                                                     0            1            everybody  103.580827   
0                   1                   1            0            0            everybody  107.677235   
                                        0            1            0            everybody  107.677235   
                                                     0            1            everybody  107.677235   
1                   0                   1            0            0            redgene    105.852329   
                                        0            1            0            redgene    105.852329   
                                                     0            1            redgene    105.852329   
0                   1                   1            0            0            redgene     30.815494   
                                        0            1            0            redgene     30.815494   
                                                     0            1            redgene     30.815494   
1                   0                   1            0            0            neurogene   15.508024   
                                        0            1            0            neurogene   15.508024   
                                                     0            1            neurogene   15.508024   
0                   1                   1            0            0            neurogene   96.310272   
                                        0            1            0            neurogene   96.310272   
                                                     0            1            neurogene   96.310272   

                                                                                                zi  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id                
1                   0                   1            0            0            nobody     0.772542   
                                        0            1            0            nobody     0.454455   
                                                     0            1            nobody     0.880184   
0                   1                   1            0            0            nobody     0.772542   
                                        0            1            0            nobody     0.454455   
                                                     0 

In [84]:
cell_counts

cells
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[1] C(rep_id)[2] C(rep_id)[3] cre_id          
1                   0                   0            1            0            nobody       503
                                                     0            1            nobody       500
                                        1            0            0            nobody       499
0                   1                   0            1            0            nobody       443
                                                     0            1            nobody       441
                                        1            0            0            nobody       440
1                   0                   0            1            0            somebody     522
                                        1            0            0            somebody     504
                                        0            0            1            somebody     498
0                   1                   1            0            0            somebody     464
                                        0            0            1            somebody     457
                                                     1            0            somebody     434
1                   0                   0            0            1            everybody    514
                                        1            0            0            everybody    501
                                        0            1            0            everybody    499
0                   1                   0            1            0            everybody    465
                                                     0            1            everybody    455
                                        1            0            0            everybody    451
1                   0                   0            0            1            redgene      523
                                        1            0            0            redgene      495
                                        0            1            0            redgene      493
0                   1                   0            1            0            redgene      465
                                                     0            1            redgene      445
                                        1            0            0            redgene      443
1                   0                   1            0            0            neurogene    499
                                        0            1            0            neurogene    491
                                                     0            1            neurogene    490
0                   1                   1            0            0            neurogene    465
                                        0            1            0            neurogene    460
                                                     0            1            neurogene    453

In [ ]:
working=cell_counts.join(flattened_param)
working["r"]=np.exp(working["theta"])
working["sigmasquare"]=working["nb"]**2/working["r"]+working["nb"]
working["p"]=working["nb"]/working["sigmasquare"]
working

cells  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] cre_id    C(rep_id)[1] C(rep_id)[2] C(rep_id)[3]          
1                   0                   1              0              nobody    1            0            0               503   
                                                                                0            1            0               503   
                                                                                             0            1               503   
                                        0              1              nobody    1            0            0               500   
                                                                                0            1            0               500   
...                                                                                                                       ...   
0                   1                   1              0              neurogene 0            1            0               460   
                                                                                             0            1               460   
                                        0              1              neurogene 1            0            0               453   
                                                                                0            1            0               453   
                                                                                             0            1               453   

                                                                                                                               nb  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] cre_id    C(rep_id)[1] C(rep_id)[2] C(rep_id)[3]              
1                   0                   1              0              nobody    1            0            0              1.770551   
                                                                                0            1            0              1.770551   
                                                                                             0            1              1.770551   
                                        0              1              nobody    1            0            0              1.770551   
                                                                                0            1            0              1.770551   
...                                                                                                                           ...   
0                   1                   1              0              neurogene 0            1            0             96.329149   
                                                                                             0            1             96.329149   
                                        0              1              neurogene 1            0            0             96.329149   
                                                                                0            1            0             96.329149   
                                                                                             0            1             96.329149   

                                                                                                                              zi  \
C(cell_type)[blood] C(cell_type)[brain] C(rep_id)[T.2] C(rep_id)[T.3] cre_id    C(rep_id)[1] C(rep_id)[2] C(rep_id)[3]             
1                   0                   1              0              nobody    1            0            0             0.773195   
                                                                                0            1            0             0.455349   
                                                                                             0            1             0.880575   
                                        0           

In [57]:
cluster.close()